In [1]:
import regex as re

#### Instead of using pre-made regex from opensource models, one should be able to create one from scratch and for that; one should know the internals. So let's start from the beginning

##### Remember the goal of a regex before tokenization is to chunk words in a meaningful manner so that BPE can only merge inside these chunks and learn meaningful merges instead of merging across word boundries.

## Let's start with a basic regex and we'll improve it one step at a time

In [11]:
samples = [
    "Hello.... world!!!!",
    "I'm learning BPE. 45320000",
    "GPT-4 costs 200,000,000.0 euros.",
    "Hallo, ich-lerne Deutsch.",
    "(Hello)   'world'\nHow are you?",
    "こんにちは世界\t\r\r\t\n",
    " hello\n\n world "
]

In [12]:
def apply_regex(regex_p, samples):
    pattern = re.compile(regex_p)
    for sample in samples:
        print(repr(sample), "->", pattern.findall(sample))



In [13]:
pattern = r'\p{L}+' # one or more unicode letters
apply_regex(pattern, samples)

'Hello.... world!!!!' -> ['Hello', 'world']
"I'm learning BPE. 45320000" -> ['I', 'm', 'learning', 'BPE']
'GPT-4 costs 200,000,000.0 euros.' -> ['GPT', 'costs', 'euros']
'Hallo, ich-lerne Deutsch.' -> ['Hallo', 'ich', 'lerne', 'Deutsch']
"(Hello)   'world'\nHow are you?" -> ['Hello', 'world', 'How', 'are', 'you']
'こんにちは世界\t\r\r\t\n' -> ['こんにちは世界']
' hello\n\n world ' -> ['hello', 'world']


#### At this point we are able to catch words. what about the numbers? punctuations? contractions like i'm? spaces, newline?

similar to the one which catches letters we also have a pattern to catch unicode numbers and spaces i.e '\p{N}+'. for whitespace char '\s+'. + means one or more here

In [14]:
pattern = r'\p{N}+'
apply_regex(pattern, samples)

'Hello.... world!!!!' -> []
"I'm learning BPE. 45320000" -> ['45320000']
'GPT-4 costs 200,000,000.0 euros.' -> ['4', '200', '000', '000', '0']
'Hallo, ich-lerne Deutsch.' -> []
"(Hello)   'world'\nHow are you?" -> []
'こんにちは世界\t\r\r\t\n' -> []
' hello\n\n world ' -> []


In [15]:
pattern = r'\s+' # white spaces and newline characters
apply_regex(pattern, samples)

'Hello.... world!!!!' -> [' ']
"I'm learning BPE. 45320000" -> [' ', ' ', ' ']
'GPT-4 costs 200,000,000.0 euros.' -> [' ', ' ', ' ']
'Hallo, ich-lerne Deutsch.' -> [' ', ' ']
"(Hello)   'world'\nHow are you?" -> ['   ', '\n', ' ', ' ']
'こんにちは世界\t\r\r\t\n' -> ['\t\r\r\t\n']
' hello\n\n world ' -> [' ', '\n\n ', ' ']


#### Great! now we can capture spaces, letters, numbers. what about Punctuations? We can combine and negate all of the above patterns to find any unicode characters that are not numbers, letters or white spaces

In [16]:
pattern = r"[^\s\p{L}\p{N}]+"
apply_regex(pattern, samples)

'Hello.... world!!!!' -> ['....', '!!!!']
"I'm learning BPE. 45320000" -> ["'", '.']
'GPT-4 costs 200,000,000.0 euros.' -> ['-', ',', ',', '.', '.']
'Hallo, ich-lerne Deutsch.' -> [',', '-', '.']
"(Hello)   'world'\nHow are you?" -> ['(', ')', "'", "'", '?']
'こんにちは世界\t\r\r\t\n' -> []
' hello\n\n world ' -> []


#### Now let's merge all the above patterns

In [17]:
pattern = r'\s+|\p{L}+|\p{N}+|[^\s\p{L}\p{N}]+'
apply_regex(pattern, samples)

'Hello.... world!!!!' -> ['Hello', '....', ' ', 'world', '!!!!']
"I'm learning BPE. 45320000" -> ['I', "'", 'm', ' ', 'learning', ' ', 'BPE', '.', ' ', '45320000']
'GPT-4 costs 200,000,000.0 euros.' -> ['GPT', '-', '4', ' ', 'costs', ' ', '200', ',', '000', ',', '000', '.', '0', ' ', 'euros', '.']
'Hallo, ich-lerne Deutsch.' -> ['Hallo', ',', ' ', 'ich', '-', 'lerne', ' ', 'Deutsch', '.']
"(Hello)   'world'\nHow are you?" -> ['(', 'Hello', ')', '   ', "'", 'world', "'", '\n', 'How', ' ', 'are', ' ', 'you', '?']
'こんにちは世界\t\r\r\t\n' -> ['こんにちは世界', '\t\r\r\t\n']
' hello\n\n world ' -> [' ', 'hello', '\n\n ', 'world', ' ']


#### Right now i'm maps too i, ',' and 'm' separately but this is one word so let;s handle that

In [18]:
pattern = r"(?i:'s|'t|'re|'ve|'m|'ll|'d)" # case-insensitive match for common endings in contraptions
apply_regex(pattern, samples)

'Hello.... world!!!!' -> []
"I'm learning BPE. 45320000" -> ["'m"]
'GPT-4 costs 200,000,000.0 euros.' -> []
'Hallo, ich-lerne Deutsch.' -> []
"(Hello)   'world'\nHow are you?" -> []
'こんにちは世界\t\r\r\t\n' -> []
' hello\n\n world ' -> []


In [19]:
# now let's merge it with our other patterns
# (?i:...) makes the regex inside the parentheses case-insensitive.
pattern = r"(?i:'s|'t|'re|'ve|'m|'ll|'d)|\s+|\p{L}+|\p{N}+|[^\s\p{L}\p{N}]+" # regex patterns are matched from left to right so contraptions get a chance to be merged before other words
apply_regex(pattern, samples)

'Hello.... world!!!!' -> ['Hello', '....', ' ', 'world', '!!!!']
"I'm learning BPE. 45320000" -> ['I', "'m", ' ', 'learning', ' ', 'BPE', '.', ' ', '45320000']
'GPT-4 costs 200,000,000.0 euros.' -> ['GPT', '-', '4', ' ', 'costs', ' ', '200', ',', '000', ',', '000', '.', '0', ' ', 'euros', '.']
'Hallo, ich-lerne Deutsch.' -> ['Hallo', ',', ' ', 'ich', '-', 'lerne', ' ', 'Deutsch', '.']
"(Hello)   'world'\nHow are you?" -> ['(', 'Hello', ')', '   ', "'", 'world', "'", '\n', 'How', ' ', 'are', ' ', 'you', '?']
'こんにちは世界\t\r\r\t\n' -> ['こんにちは世界', '\t\r\r\t\n']
' hello\n\n world ' -> [' ', 'hello', '\n\n ', 'world', ' ']


#### Notice that every type of whitespace is treated the same, however each escape char has a different meaning and it's position also matters eg. punctuation inside the sentences and at the end mean different things.

#### we have all the building blocks for a tokenizer next things are adding sophisticated rules like merging leading spaces, merging numbers into groups , merging sentence ending punctuations. The goal of this exp was to develop intuition for regex


In [25]:
# for gpt-style regex here is the structure
# | Order | Regex | Purpose |
# |---|---|---|
# | 1 | '(?i:[sdmt]|ll|ve|re) | Contractions |
# | 2 | [^\r\n\p{L}\p{N}]?+\p{L}+ | Letters with optional preceding character |
# | 3 | \p{N}{1,3} | Numeric groups of up to 3 |
# | 4 |  ?[^\s\p{L}\p{N}]++[\r\n]* | Punctuation and symbols |
# | 5 | \s*[\r\n] | Newline-ending whitespace |
# | 6 | \s+(?!\S) | Trailing whitespace / whitespace before another whitespace |
# | 7 | \s+ | General whitespace fallback |